# 6 · Day 4 – End-to-end pipeline validation
**Job task `validate`** – runs after `gold_build`. It checks the whole chain **Bronze → Silver → Gold → Mart** in one place.

Notebooks 1–5 each validate their own layer. This notebook checks that the layers **agree with each other** in the same run.
Results are appended to **`ops.pipeline_validation_results`** (one row per check per run).

| Status | Meaning |
|---|---|
| PASS | check met |
| FAIL | check not met → the notebook stops and the job is marked failed |
| WARN | worth looking at, but does not stop the job |

## 1. Setup

This cell gets the notebook ready:
* Reads the job values: `catalog`, `run_id`, `freshness_hours` (how recent Gold must be) and `max_reject_pct` (reject % limit before a warning). Defaults are used when you run it by hand.
* Sets the schema names for bronze, silver, gold and ops, and creates the `ops` schema if it does not exist.
* Defines helpers: `count()` returns a table's row count, `hours_since_last_write()` returns hours since a table was last written, and `check()` compares an actual value with the expected one, marks it PASS / FAIL / WARN, and stores the result.

In [0]:
from pyspark.sql import functions as F

# Job parameters (the job fills these in; defaults are used when you run the notebook by hand)
dbutils.widgets.text("catalog", "capstone_5iot")
dbutils.widgets.text("run_id", "manual")
CATALOG = dbutils.widgets.get("catalog")
RUN_ID  = dbutils.widgets.get("run_id")

BRONZE = f"{CATALOG}.iot_schema"
SILVER = f"{CATALOG}.silver"
GOLD   = f"{CATALOG}.gold"
OPS    = f"{CATALOG}.ops"            # pipeline monitoring tables live here
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {OPS}")

def count(table):
    return spark.table(table).count()

def hours_since_last_write(table):
    ts  = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").first()["timestamp"]
    now = spark.sql("SELECT current_timestamp()").first()[0]   # Spark clock, same time zone as ts
    return round((now - ts).total_seconds() / 3600, 2)

print(f"catalog={CATALOG}  run_id={RUN_ID}")

dbutils.widgets.text("freshness_hours", "24")      # Gold must have been written within this many hours
dbutils.widgets.text("max_reject_pct", "25")       # warn if more than this % of a Bronze table is rejected
FRESHNESS_HOURS = float(dbutils.widgets.get("freshness_hours"))
MAX_REJECT_PCT  = float(dbutils.widgets.get("max_reject_pct"))

results = []
def check(layer, name, actual, expected, on_mismatch="FAIL"):
    status = "PASS" if actual == expected else on_mismatch
    results.append((layer, name, str(actual), str(expected), status))
    print(f"[{status}] {layer} | {name}: actual={actual} expected={expected}")

## 2. Bronze – every source table landed

This cell checks that each of the 7 Bronze source tables has at least one row. An empty table is marked FAIL.

In [0]:
BRONZE_TABLES = ["bronze_machine_master", "bronze_plant_master", "bronze_failure_code_lookup",
                 "bronze_sensor_readings", "bronze_failure_events", "bronze_work_orders",
                 "bronze_work_order_status_cdc"]
for t in BRONZE_TABLES:
    check("bronze", f"{t} has rows", count(f"{BRONZE}.{t}") > 0, True)

## 3. Bronze → Silver – every Bronze record is accounted for
`Bronze rows = Silver rows + rejected rows` (rejected includes removed duplicates, rule DQ05).

This cell checks that no records were lost between Bronze and Silver:
* For sensor readings, failure events and work orders, checks that Bronze rows = Silver rows + rejected rows.
* Works out the reject % (not counting removed duplicates) and gives a WARN if it is above `max_reject_pct`.
* Checks that every row in `silver_load_audit` is reconciled.
* Checks that the Bronze row counts saved in the audit table match the current Bronze tables, which shows Silver was built from today's Bronze data.

In [0]:
rej = spark.table(f"{SILVER}.silver_rejected_records")
PAIRS = [("bronze_sensor_readings", "silver_sensor_readings"),
         ("bronze_failure_events",  "silver_failure_events"),
         ("bronze_work_orders",     "silver_work_orders")]

for b, s in PAIRS:
    bronze_rows   = count(f"{BRONZE}.{b}")
    silver_rows   = count(f"{SILVER}.{s}")
    rejected_rows = rej.filter(F.col("source_table") == b).count()
    check("bronze->silver", f"{b}: bronze = silver + rejected", bronze_rows, silver_rows + rejected_rows)

    real_rejects = rej.filter((F.col("source_table") == b) & (F.col("rule_id") != "DQ05")).count()
    pct = round(100 * real_rejects / bronze_rows, 2)
    check("bronze->silver", f"{b}: reject % <= {MAX_REJECT_PCT}", pct <= MAX_REJECT_PCT, True, on_mismatch="WARN")

# the Day 2 audit table must agree with today's Bronze (proves Silver ran on the current Bronze data)
audit = spark.table(f"{SILVER}.silver_load_audit")
check("bronze->silver", "silver_load_audit: all reconciled", audit.filter("NOT reconciled").count(), 0)
for b, _ in PAIRS:
    audited = audit.filter(F.col("bronze_table") == b).first()["bronze_rows"]
    check("bronze->silver", f"{b}: audit rows = current Bronze rows", audited, count(f"{BRONZE}.{b}"))

## 4. Silver → Gold – facts match Silver

This cell checks that each Gold table has the same number of rows as the Silver table it is built from:
* `fact_sensor_activity` vs `silver_sensor_readings`
* `fact_maintenance_episodes` vs `silver_failure_events`
* `fact_status_transitions` vs `silver_work_order_status_history`
* `dim_machine` vs `silver_machine_master`

In [0]:
check("silver->gold", "fact_sensor_activity = silver_sensor_readings",
      count(f"{GOLD}.fact_sensor_activity"), count(f"{SILVER}.silver_sensor_readings"))
check("silver->gold", "fact_maintenance_episodes = silver_failure_events",
      count(f"{GOLD}.fact_maintenance_episodes"), count(f"{SILVER}.silver_failure_events"))
check("silver->gold", "fact_status_transitions = silver_work_order_status_history",
      count(f"{GOLD}.fact_status_transitions"), count(f"{SILVER}.silver_work_order_status_history"))
check("silver->gold", "dim_machine = silver_machine_master",
      count(f"{GOLD}.dim_machine"), count(f"{SILVER}.silver_machine_master"))

## 5. Gold → Mart – KPI totals match the facts

This cell checks that the daily mart totals match the Gold fact tables:
* Adds up readings, abnormal readings, failures and downtime minutes in `mart_sensor_daily`.
* Compares each total with the same number calculated from `fact_sensor_activity` and `fact_maintenance_episodes`.
* Checks that every `machine_sk` in the sensor facts exists in `dim_machine` (no orphan rows).

In [0]:
m = spark.table(f"{GOLD}.mart_sensor_daily").agg(
        F.sum("reading_count").alias("r"), F.sum("abnormal_count").alias("a"),
        F.sum("failure_count").alias("f"), F.sum("downtime_minutes").alias("d")).first()
fsa = spark.table(f"{GOLD}.fact_sensor_activity")
fme = spark.table(f"{GOLD}.fact_maintenance_episodes")

check("gold->mart", "mart readings = fact readings",   m["r"], fsa.count())
check("gold->mart", "mart abnormal = fact abnormal",   m["a"], fsa.filter("is_abnormal").count())
check("gold->mart", "mart failures = fact failures",   m["f"], fme.count())
check("gold->mart", "mart downtime = fact downtime",   m["d"], fme.agg(F.sum("downtime_minutes")).first()[0])
check("gold->mart", "machine_sk has no orphans",
      fsa.join(spark.table(f"{GOLD}.dim_machine"), "machine_sk", "left_anti").count(), 0)

## 6. Freshness – this run really rebuilt Silver and Gold

This cell checks that the Silver sensor readings table, the Gold sensor fact table and the daily mart were all written within the last `freshness_hours` hours. If any of them is older, it is marked FAIL.

In [0]:
for t in [f"{SILVER}.silver_sensor_readings", f"{GOLD}.fact_sensor_activity", f"{GOLD}.mart_sensor_daily"]:
    h = hours_since_last_write(t)
    check("freshness", f"{t} written within {FRESHNESS_HOURS:g}h (was {h}h ago)", h <= FRESHNESS_HOURS, True)

## 7. Save results & stop the job on failure

This cell saves and summarises all the checks:
* Turns the check results into a table, adds the `run_id` and run time, and appends it to `ops.pipeline_validation_results` (old runs are kept).
* Shows the results and prints how many checks were PASS, WARN and FAIL.
* Stops the notebook with an error if any check failed, so the job is marked failed.

In [0]:
run_ts = spark.sql("SELECT current_timestamp()").first()[0]
res = (spark.createDataFrame(results, "layer string, check_name string, actual string, expected string, status string")
            .withColumn("run_id", F.lit(RUN_ID))
            .withColumn("run_ts", F.lit(run_ts)))
res.write.mode("append").option("mergeSchema", "true").saveAsTable(f"{OPS}.pipeline_validation_results")
display(res)

summary = {s: sum(1 for r in results if r[4] == s) for s in ["PASS", "WARN", "FAIL"]}
print(summary)
assert summary["FAIL"] == 0, f"{summary['FAIL']} end-to-end check(s) FAILED - see ops.pipeline_validation_results"